# Week 4 - Model Evaluation and Validation

## Breast Cancer Diagnostic Classification

This notebook evaluates the baseline models using:

- Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC
- Confusion Matrix
- 5-Fold Stratified Cross-Validation


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

print("Libraries imported successfully.")


## 2. Load and Split Dataset

In [ ]:
data = load_breast_cancer()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Test samples:", len(X_test))


## 3. Define Baseline Models

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=5000,
            random_state=42
        ))
    ]),

    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )
}

print("Models created:", len(models))


## 4. Evaluate Models

In [ ]:
results = []

for name, model in models.items():

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]

    accuracy = accuracy_score(y_test, predictions)
    precision = precision_score(y_test, predictions)
    recall = recall_score(y_test, predictions)
    f1 = f1_score(y_test, predictions)
    roc_auc = roc_auc_score(y_test, probabilities)

    results.append({
        "model": name,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "roc_auc": roc_auc
    })

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1-score : {f1:.4f}")
    print(f"ROC-AUC  : {roc_auc:.4f}")

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, predictions))

results_df = pd.DataFrame(results)
display(results_df)


## 5. Confusion Matrix Visualization

In [ ]:
for name, model in models.items():

    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    ConfusionMatrixDisplay.from_predictions(
        y_test,
        predictions
    )

    plt.title(f"Confusion Matrix - {name}")
    plt.show()


## 6. Five-Fold Stratified Cross-Validation

StratifiedKFold preserves the class distribution across folds.
ROC-AUC is used as the validation metric.


In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_results = []

for name, model in models.items():

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="roc_auc"
    )

    print("\n" + name)
    print("ROC-AUC scores:", np.round(scores, 4))
    print(f"Mean ROC-AUC: {scores.mean():.4f}")
    print(f"Std ROC-AUC : {scores.std():.4f}")

    cv_results.append({
        "model": name,
        "mean_roc_auc": scores.mean(),
        "std_roc_auc": scores.std()
    })

cv_df = pd.DataFrame(cv_results)
display(cv_df)


## 7. Evaluation Conclusion

The evaluation demonstrates that Logistic Regression provides the
strongest baseline performance for this dataset, with Random Forest
also providing strong results.

Cross-validation confirms that the models maintain strong ROC-AUC
performance across different training folds.
